# Modélisation du Risque Venteux : La Tempête Martin (1999)

### Date : 17 février 2026 
### Module : Enjeux et modélisation des risques climatiques

## Auteur : Arthur DANJOU - M2 ISF

In [98]:
import geopandas as gpd
import rasterio

import numpy as np
import pandas as pd


In [99]:
src_path = "./data/TempeteMartinWGS84.tif"
communes_path = "./data/communes.gpkg"
batiments_path = "./data/batiments.gpkg"


## Partie 1 : Analyse de l'aléa et des données d'exposition

### 1.1. Chargement et exploration

In [100]:
src = rasterio.open(src_path)
communes = gpd.read_file(communes_path)
batiments = gpd.read_file(batiments_path)

print(f"CRS Raster: {src.crs}")
print(f"CRS Communes: {communes.crs}")
print(f"CRS Bâtiments: {batiments.crs}")

print("\nExtrait Table Communes:")
print(communes.describe())
print("\nExtrait Table Bâtiments:")
print(batiments.describe())


CRS Raster: EPSG:4326
CRS Communes: EPSG:2154
CRS Bâtiments: EPSG:2154

Extrait Table Communes:
         POPULATION   SUPERF_CAD             DATE_APP            DATE_CONF  \
count    630.000000   630.000000                   45                    9   
mean    1348.401587  1616.952381  1992-11-13 14:56:00  2020-03-22 10:40:00   
min       43.000000   120.000000  1970-12-01 00:00:00  2019-01-01 00:00:00   
25%      316.500000   880.000000  1973-01-01 00:00:00  2019-01-01 00:00:00   
50%      646.500000  1385.000000  1979-06-01 00:00:00  2019-01-01 00:00:00   
75%     1320.500000  2030.000000  2016-01-01 00:00:00  2019-01-01 00:00:00   
max    79961.000000  8940.000000  2025-01-01 00:00:00  2025-01-01 00:00:00   
std     3787.117325  1086.952500                  NaN                  NaN   

                  DATE_RCT  
count                  630  
mean   2022-01-01 00:00:00  
min    2022-01-01 00:00:00  
25%    2022-01-01 00:00:00  
50%    2022-01-01 00:00:00  
75%    2022-01-01 00:00:00 

### 1.2. Cartographie de la zone d'étude

![Carte de la zone d'étude](./carte%201.png)

### 1.3. Statistiques de l'aléa

In [101]:
from rasterstats import zonal_stats
from shapely.geometry import box

bbox = box(*src.bounds)
bbox_gdf = gpd.GeoDataFrame({"geometry": [bbox]}, crs=src.crs)

global_stats = zonal_stats(
    bbox_gdf, "./data/TempeteMartinWGS84.tif", stats=["min", "max", "mean", "std"],
)[0]

df_stats = pd.DataFrame([global_stats], index=["Valeurs"])
print("Statistiques descriptives de l'aléa (Martin 1999) :")
print(df_stats)


Statistiques descriptives de l'aléa (Martin 1999) :
               min        max       mean     std
Valeurs  11.906469  52.240498  19.885484  6.9541


## Partie 2 : Croisement Aléa-Territoire et Analyse de l'Exposition

### 2.1. Statistiques zonales par commune

In [102]:
from rasterio.mask import mask

if communes.crs != src.crs:
    communes = communes.to_crs(src.crs)

resultats_stats = []
nodata_val = src.nodata

for _, row in communes.iterrows():
    try:
        out_image, _ = mask(src, [row.geometry], crop=True, all_touched=True)
        data = out_image[0]

        values = data[(data != nodata_val) & (np.isfinite(data))]

        if values.size > 0:
            resultats_stats.append(
                {
                    "INSEE_COM": row["INSEE_COM"],
                    "NOM": row["NOM"],
                    "vitesse_moyenne": float(np.mean(values)),
                    "vitesse_maximale": float(np.max(values)),
                    "vitesse_minimale": float(np.min(values)),
                    "vitesse_ecart_type": float(np.std(values)),
                },
            )
    except (ValueError, RuntimeError):
        continue

communes_resultats = pd.DataFrame(resultats_stats)
print(communes_resultats.describe())
print(communes_resultats.head())


       vitesse_moyenne  vitesse_maximale  vitesse_minimale  vitesse_ecart_type
count       630.000000        630.000000        630.000000          630.000000
mean         44.006877         44.429421         43.591589            0.244895
std           3.106150          3.153961          3.088576            0.197454
min          37.909477         38.180199         37.507687            0.010203
25%          41.646152         41.868794         41.482575            0.134389
50%          43.455757         43.896006         43.031494            0.199136
75%          46.056098         46.597459         45.413697            0.299832
max          51.705040         52.240498         51.531330            2.019503
  INSEE_COM                      NOM  vitesse_moyenne  vitesse_maximale  \
0     17058    Bourcefranc-le-Chapus        50.242519         50.528202   
1     16330  Saint-Laurent-de-Cognac        43.382690         43.635235   
2     16193       Louzac-Saint-André        43.396835         43

### 2.2. Identification des communes les plus exposées

In [103]:
top_3_moyenne = communes_resultats.nlargest(3, "vitesse_moyenne")[
    ["NOM", "vitesse_moyenne"]
]
top_3_maximale = communes_resultats.nlargest(3, "vitesse_maximale")[
    ["NOM", "vitesse_maximale"]
]

print("Top 3 des communes - Vitesse de vent moyenne la plus élevée :")
print(top_3_moyenne)

print("\nTop 3 des communes - Vitesse de vent maximale la plus élevée :")
print(top_3_maximale)


Top 3 des communes - Vitesse de vent moyenne la plus élevée :
                        NOM  vitesse_moyenne
16        La Brée-les-Bains        51.705040
349  Saint-Georges-d'Oléron        51.696590
583   Saint-Pierre-d'Oléron        51.500526

Top 3 des communes - Vitesse de vent maximale la plus élevée :
                        NOM  vitesse_maximale
349  Saint-Georges-d'Oléron         52.240498
583   Saint-Pierre-d'Oléron         52.078884
459               Île-d'Aix         51.875992


### 2.3. Exposition des bâtiments

#### 2.3.1. Transformation de la couche de polygones `batiments.gpkg` en une couche de points (centroïdes).

In [104]:
batiments_pts = batiments.copy()
batiments_pts["geometry"] = batiments_pts.geometry.centroid


#### 2.3.2. Pour chaque bâtiment (point), extraction de la valeur de vitesse de vent correspondante depuis le raster `TempeteMartinWGS84.tif`.

In [105]:
batiments_pts = batiments.copy()
batiments_pts["geometry"] = batiments_pts.geometry.centroid

if batiments_pts.crs != src.crs:
    batiments_pts = batiments_pts.to_crs(src.crs)

coords = [(geom.x, geom.y) for geom in batiments_pts.geometry]

sampled = list(src.sample(coords))
batiments_pts["vitesse_vent"] = [val[0] for val in sampled]

batiments_pts["vitesse_vent"] = batiments_pts["vitesse_vent"].replace(
    src.nodata, np.nan,
)

print(
    f"Nombre de bâtiments avec une valeur de vent valide : {batiments_pts['vitesse_vent'].notna().sum()}",
)
print(batiments_pts.head())
print(batiments_pts.describe())


Nombre de bâtiments avec une valeur de vent valide : 101370
                         ID                  USAGE1  NB_ETAGES MAT_TOITS  \
0  BATIMENT0000000258101741             Résidentiel        1.0        10   
1  BATIMENT0000000292067994           Indifférencié        NaN      None   
2  BATIMENT0000000292068000  Commercial et services        2.0      None   
3  BATIMENT0000000292068017           Indifférencié        NaN      None   
4  BATIMENT0000000292068024           Indifférencié        NaN      None   

   HAUTEUR  Z_MIN_SOL  Z_MAX_TOIT                   geometry  vitesse_vent  
0      3.3       50.0        53.7  POINT (-0.50422 46.09339)     41.530777  
1      5.3        5.0         NaN   POINT (-1.17684 46.0137)     51.538124  
2      6.1        6.5        14.6  POINT (-1.17687 46.01255)     51.538124  
3      2.7        6.0         9.3    POINT (-1.17691 46.012)     51.538124  
4      3.4        6.0        11.5  POINT (-1.17672 46.01214)     51.538124  
          NB_ETAGES  

## Partie 3 : Estimation des Pertes Économiques

### 3.1. Calcul de la valeur assurée


In [113]:
cout_reconstruction_m2 = 1500

batiments["surface"] = batiments.geometry.area
batiments_pts["surface"] = batiments.geometry.area
batiments_pts["valeur_assuree"] = batiments_pts["surface"] * cout_reconstruction_m2

print("Aperçu du calcul de la valeur assurée :")
print(batiments_pts[["surface", "valeur_assuree"]].head())

print(batiments_pts.describe())

print(
    f"\nValeur totale du portefeuille assuré : {batiments_pts['valeur_assuree'].sum():,.2f} €",
)


Aperçu du calcul de la valeur assurée :
   surface  valeur_assuree
0  184.095   276142.499996
1   12.905    19357.499999
2  617.905   926857.500003
3   12.575    18862.500000
4   98.280   147420.000003
          NB_ETAGES       HAUTEUR      Z_MIN_SOL    Z_MAX_TOIT   vitesse_vent  \
count  45989.000000  99954.000000  100552.000000  87034.000000  101370.000000   
mean       1.308965      4.129201      26.883678     33.297492      46.290665   
std        0.560797      1.827434      25.609158     26.130012       3.521599   
min        0.000000      0.000000      -3.000000      0.000000      37.835918   
25%        1.000000      3.000000       7.800000     13.800000      43.306957   
50%        1.000000      3.800000      18.000000     24.500000      46.618694   
75%        2.000000      4.900000      38.000000     44.900000      49.390682   
max       14.000000     50.400000     158.300000    165.900000      52.240498   

             surface  valeur_assuree             dr  perte_economiqu

### 3.2. Application de la fonction de dommage

In [114]:
def get_damage_ratio(vitesse: float) -> float:
    """Compute the damage ratio based on wind speed."""
    if vitesse < 25:
        return 0.00
    if 25 <= vitesse < 30:
        return 0.05
    if 30 <= vitesse < 35:
        return 0.15
    if 35 <= vitesse < 40:
        return 0.30
    if 40 <= vitesse < 45:
        return 0.55
    if 45 <= vitesse < 50:
        return 0.80
    return 1.00


batiments_pts["dr"] = batiments_pts["vitesse_vent"].apply(get_damage_ratio)
batiments_pts["perte_economique"] = (
    batiments_pts["valeur_assuree"] * batiments_pts["dr"]
)

print(batiments_pts.describe())
print(batiments_pts["perte_economique"].describe())


          NB_ETAGES       HAUTEUR      Z_MIN_SOL    Z_MAX_TOIT   vitesse_vent  \
count  45989.000000  99954.000000  100552.000000  87034.000000  101370.000000   
mean       1.308965      4.129201      26.883678     33.297492      46.290665   
std        0.560797      1.827434      25.609158     26.130012       3.521599   
min        0.000000      0.000000      -3.000000      0.000000      37.835918   
25%        1.000000      3.000000       7.800000     13.800000      43.306957   
50%        1.000000      3.800000      18.000000     24.500000      46.618694   
75%        2.000000      4.900000      38.000000     44.900000      49.390682   
max       14.000000     50.400000     158.300000    165.900000      52.240498   

             surface  valeur_assuree             dr  perte_economique  \
count  101508.000000    1.015080e+05  101508.000000      1.015080e+05   
mean      110.040776    1.650612e+05       0.723555      1.162034e+05   
std       266.385398    3.995781e+05       0.176776

### 3.3. Analyse des pertes totales

In [108]:
perte_totale = batiments_pts["perte_economique"].sum()
batiments_endommages = batiments_pts[batiments_pts["perte_economique"] > 0]
nb_endommages = len(batiments_endommages)
perte_moyenne = perte_totale / nb_endommages if nb_endommages > 0 else 0

print(f"Perte économique totale : {perte_totale:,.2f} €")
print(f"Nombre total de bâtiments endommagés : {nb_endommages}")
print(f"Perte moyenne par bâtiment endommagé : {perte_moyenne:,.2f} €")


Perte économique totale : 11,795,577,774.00 €
Nombre total de bâtiments endommagés : 101508
Perte moyenne par bâtiment endommagé : 116,203.43 €


In [109]:
batiments_with_commune = gpd.sjoin(
    batiments_pts,
    communes[["INSEE_COM", "NOM", "geometry"]],
    how="left",
    predicate="within",
)

pertes_communes = (
    batiments_with_commune.groupby("INSEE_COM")["perte_economique"].sum().reset_index()
)

communes_final = communes.merge(pertes_communes, on="INSEE_COM", how="left")
communes_final["perte_economique"] = communes_final["perte_economique"].fillna(0)

communes_final.to_file(
    "./data/communes_pertes_final.gpkg", layer="pertes_par_commune", driver="GPKG",
)


![Carte de la zone d'étude](./carte%202.png)

## Partie 4 : Analyse Exploratoire et Approfondissement

### Piste 1 : Modulation de la fonction de dommage par le matériau

In [116]:
def get_vulnerability_factor_coded(mat: str | int | None) -> float:
    """Return a vulnerability factor based on the coded material type."""
    if mat is None:
        return 1.0

    mapping = {
        "10": 1.2,
        "09": 1.1,
        "12": 1.1,
        "00": 1.0,
        "90": 1.0,
        "19": 1.0,
        "29": 1.0,
        "39": 1.0,
        "49": 1.0,
        "91": 1.0,
        "21": 1.0,
        "24": 1.0,
        "41": 1.0,
        "13": 0.9,
        "14": 0.9,
        "30": 0.9,
        "40": 0.9,
        "23": 0.9,
        "34": 0.9,
        "01": 0.8,
        "02": 0.8,
        "20": 0.8,
        "03": 0.8,
        "04": 0.8,
    }

    return mapping.get(str(mat).strip(), 1.0)

batiments_pts["vulnerability_factor"] = batiments_pts["MAT_TOITS"].apply(
    get_vulnerability_factor_coded,
)
batiments_pts["perte_affinee"] = (
    batiments_pts["perte_economique"] * batiments_pts["vulnerability_factor"]
)
batiments_pts["variation_perte"] = (
    batiments_pts["perte_affinee"] - batiments_pts["perte_economique"]
)


batiments_pts["vulnerability_factor"] = batiments_pts["MAT_TOITS"].apply(
    get_vulnerability_factor_coded,
)

batiments_pts["perte_affinee"] = (
    batiments_pts["perte_economique"] * batiments_pts["vulnerability_factor"]
)

stats_mat = (
    batiments_pts.groupby("MAT_TOITS")
    .agg({"ID": "count", "perte_economique": "sum", "perte_affinee": "sum"})
    .rename(columns={"ID": "nb_batiments"})
)

stats_mat["variation_perte"] = (
    stats_mat["perte_affinee"] - stats_mat["perte_economique"]
)

print(stats_mat.sort_values(by="perte_economique", ascending=False))


           nb_batiments  perte_economique  perte_affinee  variation_perte
MAT_TOITS                                                                
10                32159      4.726775e+09   5.672130e+09     9.453550e+08
01                 4424      3.979362e+08   3.183490e+08    -7.958724e+07
00                 2851      2.812398e+08   2.812398e+08     0.000000e+00
13                  575      8.918949e+07   8.027054e+07    -8.918949e+06
90                  633      7.244555e+07   7.244555e+07     0.000000e+00
40                  213      4.847359e+07   4.362623e+07    -4.847359e+06
19                  240      3.508498e+07   3.508498e+07     0.000000e+00
09                  567      3.065386e+07   3.371924e+07     3.065386e+06
20                  224      2.967865e+07   2.374292e+07    -5.935731e+06
12                   90      1.518114e+07   1.669925e+07     1.518114e+06
30                   93      1.484553e+07   1.336098e+07    -1.484553e+06
14                   74      1.074817e

### Piste 2 : Analyse de l'impact de la hauteur des bâtiments

In [111]:
perte_totale = batiments_pts["perte_affinee"].sum()
valeur_totale = batiments_pts["valeur_assuree"].sum()
loss_ratio_global = (perte_totale / valeur_totale) * 100

nb_total_loss = len(batiments_pts[batiments_pts["dr"] >= 0.8])

p95_loss = batiments_pts["perte_affinee"].quantile(0.95)

print(f"Ratio de perte global : {loss_ratio_global:.2f}%")
print(f"Nombre de bâtiments en perte quasi-totale (DR >= 80%) : {nb_total_loss}")
print(f"PML 95% (Seuil des 5% des pertes les plus élevées) : {p95_loss:,.2f} €")


Ratio de perte global : 75.56%
Nombre de bâtiments en perte quasi-totale (DR >= 80%) : 62112
PML 95% (Seuil des 5% des pertes les plus élevées) : 347,898.31 €


### Piste 3 : Cartographie avancée et communication

In [112]:
batiments_pts["vitesse_vent_stress"] = batiments_pts["vitesse_vent"] * 1.10

batiments_pts["dr_stress"] = batiments_pts["vitesse_vent_stress"].apply(
    get_damage_ratio,
)
perte_stress = (batiments_pts["dr_stress"] * batiments_pts["valeur_assuree"]).sum()

augmentation_percent = ((perte_stress - perte_totale) / perte_totale) * 100
print(
    f"Une hausse de 10% du vent entraîne une hausse de {augmentation_percent:.2f}% des pertes.",
)


Une hausse de 10% du vent entraîne une hausse de 17.34% des pertes.
